# CA-SAM2 Knowledge Distillation
Run cells top to bottom. Connect to a **T4 GPU** runtime first:
> Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — connect a T4 runtime before continuing')

In [ ]:
# 2. Install dependencies
!pip install -q scikit-image scipy opencv-python-headless huggingface_hub Pillow timm
!pip install -q git+https://github.com/ChaoningZhang/MobileSAM.git

In [ ]:
# 3. Clone MedSAM2 + this repo
!git clone https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
!pip install -q -e /content/MedSAM2
!git clone https://github.com/elakiyasivakumar/SAM2-Coronary-Angiography-VA.git /content/SAM2

In [ ]:
# 4. Symlink /home/jupyter so distill scripts work without path changes
import os
os.makedirs('/home/jupyter', exist_ok=True)

# Point MedSAM2 symlink
if not os.path.exists('/home/jupyter/MedSAM2'):
    os.symlink('/content/MedSAM2', '/home/jupyter/MedSAM2')

In [ ]:
# 5. Authenticate with Google Cloud to access GCS bucket
from google.colab import auth
auth.authenticate_user()
!gcloud config set project project-8d13bdd5-5a56-45eb-934

In [ ]:
# 6. Download ARCADE train + val from GCS (~50MB)
!mkdir -p /home/jupyter/arcade_train/images /home/jupyter/arcade_train/masks
!mkdir -p /home/jupyter/arcade_val/images   /home/jupyter/arcade_val/masks

!gsutil -m cp -r gs://coronary-angio-v2/datasets/arcade/train/images/* /home/jupyter/arcade_train/images/
!gsutil -m cp -r gs://coronary-angio-v2/datasets/arcade/train/masks/*  /home/jupyter/arcade_train/masks/
!gsutil -m cp -r gs://coronary-angio-v2/datasets/arcade/val/images/*   /home/jupyter/arcade_val/images/
!gsutil -m cp -r gs://coronary-angio-v2/datasets/arcade/val/masks/*    /home/jupyter/arcade_val/masks/

import glob
print(f'Train: {len(glob.glob("/home/jupyter/arcade_train/images/*.png"))} images')
print(f'Val:   {len(glob.glob("/home/jupyter/arcade_val/images/*.png"))} images')

In [ ]:
# 7. Download teacher checkpoint from HuggingFace
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='Elakiya17/CA-SAM2', filename='medsam2_arcade_v2.pt',
                local_dir='/home/jupyter')
print('Teacher checkpoint ready.')

In [ ]:
# 8. Train MobileSAM student
# Soft labels generated on-the-fly from HF teacher (~5-10 min), then 30 epochs training
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'distill_student.py', '--student', 'mobilesam', '--ablation', '4'],
    cwd='/content/SAM2/distill',
    env={**__import__('os').environ, 'PYTHONPATH': '/content/MedSAM2'},
)
print('Exit code:', result.returncode)

In [ ]:
# 9. Train RepViT-SAM student (reuses cached soft labels from step above)
result = subprocess.run(
    [sys.executable, 'distill_student.py', '--student', 'repvitsam', '--ablation', '4'],
    cwd='/content/SAM2/distill',
    env={**__import__('os').environ, 'PYTHONPATH': '/content/MedSAM2'},
)
print('Exit code:', result.returncode)

In [ ]:
# 10. Check results
import json, glob
for f in glob.glob('/home/jupyter/results_*.json'):
    with open(f) as fp:
        r = json.load(fp)
    print(f"{r['student']} abl{r['ablation']}: Dice={r['dice_mean']:.3f}±{r['dice_std']:.3f}  IoU={r['iou_mean']:.3f}")